In [1]:
import torch
import torch.nn as nn
import numpy as np
import cv2

In [ ]:
img = cv2.imread(r"E:\mastercode\data\shr_watermelon\images\train\dsc00001.jpg" , cv2.IMREAD_UNCHANGED)

In [ ]:
img.shape # type: ignore

(640, 960, 3)

In [1]:
import os
import json
import argparse
from pathlib import Path
from PIL import Image  # pip install Pillow

In [1]:
"""
YOLO TXT → COCO JSON 一键转换脚本
用法：修改下方配置区 → 直接运行 python yolo2coco.py
纯标准库，无需 pip install 任何东西
"""

# ===================== ✏️ 配置区（只改这里） =====================

# 类别名称列表（按 YOLO class_id 顺序填写）
CLASS_NAMES = ["blooming male", "unknown", "closed male", "blooming female", "closed female"]

# YOLO 标注文件夹（里面是 .txt 文件）
LABELS_DIR = "E:\\mastercode\\data\\shr_watermelon\\object_detection\\labels\\val"

# 图片文件夹（里面是图片文件）
IMAGES_DIR = "E:\\mastercode\\data\\shr_watermelon\\object_detection\\images\\val"

# 输出 COCO JSON 文件路径
OUTPUT_JSON = "E:\\mastercode\\data\\shr_watermelon\\object_detection\\images\\val\\annotations.json"

# ================================================================


import os
import json
import subprocess
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}


# ───────────── 获取图片尺寸（Windows 专用，绝对不报错） ─────────────
def get_image_size(filepath: str):
    """利用 Windows 自带的 PowerShell 获取图片宽高，无需安装任何库"""
    abs_path = os.path.abspath(filepath).replace("'", "''")
    ps_cmd = (
        f"Add-Type -AssemblyName System.Drawing; "
        f"$img = [System.Drawing.Image]::FromFile('{abs_path}'); "
        f"Write-Output ($img.Width.ToString() + ' ' + $img.Height.ToString()); "
        f"$img.Dispose()"
    )
    result = subprocess.run(
        ["powershell", "-NoProfile", "-Command", ps_cmd],
        capture_output=True, text=True, creationflags=0x08000000  # 隐藏黑框
    )
    if result.returncode == 0 and result.stdout.strip():
        w, h = result.stdout.strip().split()
        return int(w), int(h)
    raise ValueError(f"无法获取尺寸: {filepath}")


# ───────────── 转换主逻辑 ─────────────
def yolo_to_coco():
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": i, "name": name, "supercategory": "object"}
            for i, name in enumerate(CLASS_NAMES)
        ],
    }

    label_files = sorted(Path(LABELS_DIR).glob("*.txt"))
    if not label_files:
        print(f"❌ 在 {LABELS_DIR} 中没找到 .txt 文件")
        return

    ann_id = 1
    img_id = 1

    for lf in label_files:
        stem = lf.stem

        # 自动匹配图片
        img_path = None
        for p in Path(IMAGES_DIR).iterdir():
            if p.stem == stem and p.suffix.lower() in IMAGE_EXTS:
                img_path = str(p)
                break

        if not img_path:
            print(f"⚠️  找不到图片: {stem}.*  跳过")
            continue

        try:
            img_w, img_h = get_image_size(img_path)
        except Exception as e:
            print(f"⚠️  {e}  跳过")
            continue

        coco["images"].append({
            "id": img_id,
            "file_name": Path(img_path).name,
            "width": img_w,
            "height": img_h,
        })

        for line in lf.read_text(encoding="utf-8").strip().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cid  = int(parts[0])
            xc   = float(parts[1])
            yc   = float(parts[2])
            bw   = float(parts[3])
            bh   = float(parts[4])

            abs_w = bw * img_w
            abs_h = bh * img_h
            abs_x = xc * img_w - abs_w / 2
            abs_y = yc * img_h - abs_h / 2

            abs_x = max(0.0, abs_x)
            abs_y = max(0.0, abs_y)
            abs_w = min(abs_w, img_w - abs_x)
            abs_h = min(abs_h, img_h - abs_y)

            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cid,
                "bbox": [round(abs_x, 2), round(abs_y, 2),
                         round(abs_w, 2), round(abs_h, 2)],
                "area": round(abs_w * abs_h, 2),
                "iscrowd": 0,
                "segmentation": [],
            })
            ann_id += 1

        img_id += 1

    # 保存
    os.makedirs(os.path.dirname(OUTPUT_JSON) or ".", exist_ok=True)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成！")
    print(f"   图片: {len(coco['images'])} 张")
    print(f"   标注: {len(coco['annotations'])} 个")
    print(f"   输出: {OUTPUT_JSON}")


if __name__ == "__main__":
    yolo_to_coco()

✅ 完成！
   图片: 620 张
   标注: 1486 个
   输出: E:\mastercode\data\shr_watermelon\object_detection\images\val\annotations.json


In [4]:
"""
YOLO 分割 TXT → COCO 分割 JSON 一键转换脚本
用法：修改下方配置区 → 直接运行 python yolo2coco_seg.py
"""

# ===================== ✏️ 配置区（只改这里） =====================

# 类别名称列表（按 YOLO class_id 顺序填写）
CLASS_NAMES = ["blooming male", "unknown", "closed male", "blooming female", "closed female"]  # 改成你的类别

# YOLO 分割标注文件夹（里面是 .txt 文件，格式: class x1 y1 x2 y2 ...）
LABELS_DIR = r"E:\mastercode\data\shr_watermelon\segmentation\labels\train"

# 图片文件夹（里面是图片文件）
IMAGES_DIR = r"E:\mastercode\data\shr_watermelon\segmentation\images\train"

# 输出 COCO JSON 文件路径
OUTPUT_JSON = r"E:\mastercode\data\shr_watermelon\segmentation\images\annotations_seg_train.json"

# ================================================================

import os
import json
import subprocess
from pathlib import Path

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}


# ───────────── 获取图片尺寸（Windows 专用） ─────────────
def get_image_size(filepath: str):
    abs_path = os.path.abspath(filepath).replace("'", "''")
    ps_cmd = (
        f"Add-Type -AssemblyName System.Drawing; "
        f"$img = [System.Drawing.Image]::FromFile('{abs_path}'); "
        f"Write-Output ($img.Width.ToString() + ' ' + $img.Height.ToString()); "
        f"$img.Dispose()"
    )
    result = subprocess.run(
        ["powershell", "-NoProfile", "-Command", ps_cmd],
        capture_output=True, text=True, creationflags=0x08000000 
    )
    if result.returncode == 0 and result.stdout.strip():
        w, h = result.stdout.strip().split()
        return int(w), int(h)
    raise ValueError("无法获取尺寸")


# ───────────── 数学计算：多边形面积（鞋带公式） ─────────────
def calculate_polygon_area(pts):
    """计算多边形面积，pts 为 [(x1,y1), (x2,y2), ...]"""
    n = len(pts)
    if n < 3:
        return 0.0
    area = 0.0
    for i in range(n):
        j = (i + 1) % n
        area += pts[i][0] * pts[j][1]
        area -= pts[j][0] * pts[i][1]
    return abs(area) / 2.0


# ───────────── 转换主逻辑 ─────────────
def yolo_seg_to_coco():
    coco = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": i, "name": name, "supercategory": "object"}
            for i, name in enumerate(CLASS_NAMES)
        ],
    }

    label_files = sorted(Path(LABELS_DIR).glob("*.txt"))
    total_files = len(label_files)
    
    if total_files == 0:
        print(f"❌ 在 {LABELS_DIR} 中没找到 .txt 文件")
        return

    ann_id = 1
    img_id = 1
    success_count = 0

    print(f"🚀 开始转换【分割数据】，共发现 {total_files} 个标注文件...\n")

    for idx, lf in enumerate(label_files, 1):
        stem = lf.stem

        # 1. 自动匹配图片
        img_path = None
        for p in Path(IMAGES_DIR).iterdir():
            if p.stem == stem and p.suffix.lower() in IMAGE_EXTS:
                img_path = str(p)
                break

        if not img_path:
            print(f"[{idx}/{total_files}] ⚠️  找不到图片: {stem}.* (跳过)")
            continue

        # 2. 获取图片尺寸
        try:
            img_w, img_h = get_image_size(img_path)
        except Exception:
            print(f"[{idx}/{total_files}] ⚠️  无法读取尺寸: {Path(img_path).name} (跳过)")
            continue

        # 3. 添加图片信息
        coco["images"].append({
            "id": img_id,
            "file_name": Path(img_path).name,
            "width": img_w,
            "height": img_h,
        })

        # 4. 解析分割标注
        current_img_anns = 0
        for line in lf.read_text(encoding="utf-8").strip().splitlines():
            parts = line.strip().split()
            # 至少需要 class_id + 3个点(6个坐标) = 7个元素
            if len(parts) < 7: 
                continue

            cid = int(parts[0])
            coords = [float(c) for c in parts[1:]]
            
            # 坐标必须是成对的 (x, y)
            if len(coords) % 2 != 0:
                continue 

            # 归一化 -> 绝对像素坐标
            seg_flat = []      # COCO 需要的展平格式 [x1, y1, x2, y2...]
            poly_pts = []      # 用于计算面积和 bbox 的元组格式 [(x1,y1), ...]
            
            for i in range(0, len(coords), 2):
                px = coords[i] * img_w
                py = coords[i+1] * img_h
                
                # 防止坐标越界
                px = max(0.0, min(px, float(img_w)))
                py = max(0.0, min(py, float(img_h)))
                
                seg_flat.extend([round(px, 2), round(py, 2)])
                poly_pts.append((px, py))

            # 计算外接矩形 bbox [x_min, y_min, width, height]
            xs = [p[0] for p in poly_pts]
            ys = [p[1] for p in poly_pts]
            x_min = min(xs)
            y_min = min(ys)
            bbox_w = max(xs) - x_min
            bbox_h = max(ys) - y_min

            # 计算多边形真实面积
            area = calculate_polygon_area(poly_pts)

            # 过滤掉面积或宽高为 0 的无效标注
            if area <= 0 or bbox_w <= 0 or bbox_h <= 0:
                continue

            coco["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cid,
                "segmentation": [seg_flat],  # COCO 格式要求外层套一个列表
                "bbox": [round(x_min, 2), round(y_min, 2), 
                         round(bbox_w, 2), round(bbox_h, 2)],
                "area": round(area, 2),
                "iscrowd": 0
            })
            ann_id += 1
            current_img_anns += 1

        # 5. 打印单张图片处理结果
        print(f"[{idx}/{total_files}] ✅ 搞定: {Path(img_path).name:<40} | 尺寸: {img_w}x{img_h} | 分割实例: {current_img_anns}")
        
        img_id += 1
        success_count += 1

    # 6. 保存 JSON
    os.makedirs(os.path.dirname(OUTPUT_JSON) or ".", exist_ok=True)
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(coco, f, ensure_ascii=False, indent=2)

    print(f"\n🎉 全部完成！")
    print(f"   成功处理: {success_count} / {total_files} 张图片")
    print(f"   总分割实例: {len(coco['annotations'])} 个")
    print(f"   输出文件: {OUTPUT_JSON}")


if __name__ == "__main__":
    yolo_seg_to_coco()

🚀 开始转换【分割数据】，共发现 2472 个标注文件...

[1/2472] ✅ 搞定: dsc00001.jpg                             | 尺寸: 960x640 | 分割实例: 4
[2/2472] ✅ 搞定: dsc00001_flipped.jpg                     | 尺寸: 960x640 | 分割实例: 4
[3/2472] ✅ 搞定: dsc00001_TOP.jpg                         | 尺寸: 960x640 | 分割实例: 4
[4/2472] ✅ 搞定: dsc00002.jpg                             | 尺寸: 960x640 | 分割实例: 4
[5/2472] ✅ 搞定: dsc00002_flipped.jpg                     | 尺寸: 960x640 | 分割实例: 3
[6/2472] ✅ 搞定: dsc00003.jpg                             | 尺寸: 960x640 | 分割实例: 4
[7/2472] ✅ 搞定: dsc00003_flipped.jpg                     | 尺寸: 960x640 | 分割实例: 4
[8/2472] ✅ 搞定: dsc00004.jpg                             | 尺寸: 960x640 | 分割实例: 3
[9/2472] ✅ 搞定: dsc00004_brightness_flipped.jpg          | 尺寸: 960x640 | 分割实例: 3
[10/2472] ✅ 搞定: dsc00004_brightness_flipped_TOP.jpg      | 尺寸: 960x640 | 分割实例: 3
[11/2472] ✅ 搞定: dsc00004_brightness_TOP.jpg              | 尺寸: 960x640 | 分割实例: 3
[12/2472] ✅ 搞定: dsc00004_flipped.jpg                     | 尺寸: 960x640 | 分割实例: 3
[13/2